# 83. Release the three final adapters on Hugging Face

**What this notebook does.** Publishes the paper's three final LoRA adapters (Qwen2.5-3B v13 from notebook 73e, Qwen2.5-7B v12 from notebook 73f, Llama-3.1-8B v13 in the input role from notebook 73c) to the Hugging Face Hub, each with a model card whose numbers are read from the committed report files, and records the resulting URLs and commit hashes in `reports/release/adapters_83.csv`.

**What it needs.** A Hugging Face **write** token, entered at run time in Stage B (it is held in memory only and never written to Drive). No model is loaded and nothing is generated: the notebook is CPU-only and costs nothing.

**What it writes.** `reports/release/cards/<repo-name>/README.md` (the three model cards, committed so the repository holds what was published) and `reports/release/adapters_83.csv`.

**Licences.** Each adapter is a derivative of its base model and is released under that base's licence: Qwen2.5-7B-Instruct is Apache 2.0; Qwen2.5-3B-Instruct is under the Qwen Research License (non-commercial); Llama-3.1-8B-Instruct is under the Llama 3.1 Community License, which requires "Built with Llama" on the card and a model name beginning with "Llama".

Stages: A setup and checks (CPU) · B Hugging Face login · C model cards from the report files · D preview · E upload and verify · F record · commit.

In [ ]:
# === Stage A. Setup: Drive, repository, the three adapters, checks (CPU only) ===
import os, json, re, csv, hashlib, subprocess, sys, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
NB = '83'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=0.24'], check=True)
from huggingface_hub import HfApi, login, hf_hub_download

TASK_SYSTEM = "You are a helpful assistant that processes the content users give you."   # src/targets.py
PAPER = 'Twin Distillation Against Indirect Prompt Injection: Training Language Models to Distinguish User Instructions from Injected Instructions'
REPO_URL = 'https://github.com/anasbiswas1/picalib-research'
COMMIT_NUMBERS = '08e477d'     # the commit whose report files give every number below

# the three finals: adapter directory on Drive, base model, report keys, licence, Hub repo name
FINALS = [
    dict(tag='q3_v13', name='Qwen2.5-3B-Instruct-TwinDistillation', base='Qwen/Qwen2.5-3B-Instruct',
         adapter='data/distill13_3b/lora_3b', layout='one user turn: task sentence, blank line, document',
         sec_file='reports/forged/security_73e.csv', sec_row='v13 = v7 + forged-turn phase with forged-turn processing examples', base_row='Qwen2.5-3B undefended',
         license='other', license_name='qwen-research', license_link='https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/blob/main/LICENSE',
         notebook='73e', tokenizer=None),
    dict(tag='q7_v12', name='Qwen2.5-7B-Instruct-TwinDistillation', base='Qwen/Qwen2.5-7B-Instruct',
         adapter='data/distill7b_v12/lora_7b', layout='one user turn: task sentence, blank line, document',
         sec_file='reports/forged/security_73f.csv', sec_row='v12 = v10 + forged-turn phase with forged-turn processing examples', base_row='Qwen2.5-7B undefended',
         license='apache-2.0', license_name=None, license_link=None, notebook='73f', tokenizer=None),
    dict(tag='l_role_v13', name='Llama-3.1-8B-Instruct-TwinDistillation', base='meta-llama/Llama-3.1-8B-Instruct',
         adapter='data/llama_role_v13/lora', layout="two roles, as Meta-SecAlign specifies: the task in the user role, the document in the input role (tokenizer and chat template of facebook/Meta-SecAlign-8B)",
         sec_file='reports/forged/security_73c.csv', sec_row='Llama v13 = notebook 60 + forged-turn phase', base_row='undefended Llama-3.1-8B (one user turn)',
         license='llama3.1', license_name=None, license_link='https://github.com/meta-llama/llama-models/blob/main/models/llama3_1/LICENSE',
         notebook='73c', tokenizer='facebook/Meta-SecAlign-8B'),
]
KEEP = ('adapter_config.json', 'adapter_model.safetensors', 'adapter_model.bin')

def sha256(p, n=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(n), b''): h.update(b)
    return h.hexdigest()

for f in FINALS:
    d = Path(f['adapter']); assert d.is_dir(), f'adapter directory missing: {d}'
    files = [p for p in d.iterdir() if p.name in KEEP]
    assert any(p.name == 'adapter_config.json' for p in files), f'no adapter_config.json in {d}'
    assert any(p.name.startswith('adapter_model') for p in files), f'no adapter weights in {d}'
    cfg = json.load(open(d / 'adapter_config.json'))
    f['files'] = files; f['cfg'] = cfg
    f['size_mb'] = sum(p.stat().st_size for p in files) / 1e6
    f['sha'] = {p.name: sha256(p) for p in files}
    print(f"{f['tag']:11s} {str(d):32s} r={cfg.get('r')} alpha={cfg.get('lora_alpha')} base={cfg.get('base_model_name_or_path')} {f['size_mb']:.1f} MB")
    assert cfg.get('r') == 16 and cfg.get('lora_alpha') == 32, 'adapter rank or alpha differs from the paper (rank 16, alpha 32)'
print('Stage A done: three adapters found and checked')

In [ ]:
# === Stage B. Hugging Face login (write token, kept in memory only) ===
from getpass import getpass
tok_ = getpass('Hugging Face write token (hf_...): ').strip()
assert tok_.startswith('hf_'), 'that is not a Hugging Face token'
login(token=tok_, add_to_git_credential=False)
api = HfApi(token=tok_)
HF_USER = api.whoami()['name']
for f in FINALS: f['repo_id'] = f'{HF_USER}/{f["name"]}'
print('logged in as', HF_USER); print('\n'.join(f['repo_id'] for f in FINALS))
del tok_

In [ ]:
# === Stage C. Model cards, every number read from the committed report files ===
import pandas as pd
NUM = re.compile(r'([+-]?\d*\.?\d+)\s*(?:\[\s*([+-]?\d*\.?\d+)\s*,\s*([+-]?\d*\.?\d+)\s*\])?')
def cell(df, col, value, column):
    rows = df[df[col] == value]
    assert len(rows) == 1, (value, len(rows)); return str(rows.iloc[0][column])
def point(s): return NUM.search(s).group(1)

S78 = pd.read_csv('reports/final/security_78.csv'); U78 = pd.read_csv('reports/final/user_issued_78.csv'); G79 = pd.read_csv('reports/baselines/gcg_79.csv')
BASE_TAG = {'q3_v13': 'q3_undefended', 'q7_v12': 'q7_undefended', 'l_role_v13': 'l_undefended'}
for f in FINALS:
    S = pd.read_csv(f['sec_file'])
    f['num'] = dict(
        fixed13=point(cell(S, 'model', f['sec_row'], 'fixed_13')),      fixed13_base=point(cell(S, 'model', f['base_row'], 'fixed_13')),
        adaptive=point(cell(S, 'model', f['sec_row'], 'adaptive_13')),  adaptive_base=point(cell(S, 'model', f['base_row'], 'adaptive_13')),
        combined=point(cell(S, 'model', f['sec_row'], 'combined')),     combined_base=point(cell(S, 'model', f['base_row'], 'combined')),
        s2=point(cell(S78, 'condition', f['tag'], 'fresh_injections_rule')),   s2_base=point(cell(S78, 'condition', BASE_TAG[f['tag']], 'fresh_injections_rule')),
        s4=point(cell(U78, 'condition', f['tag'], 'user_issued_fresh_240')),   s4_base=point(cell(U78, 'condition', BASE_TAG[f['tag']], 'user_issued_fresh_240')),
        gcg=point(cell(G79, 'condition', f['tag'], 'broken')),                 gcg_base=point(cell(G79, 'condition', BASE_TAG[f['tag']], 'broken')))

def card(f):
    n = f['num']; yaml = ['---', f"license: {f['license']}"]
    if f['license_name']: yaml.append(f"license_name: {f['license_name']}")
    if f['license_link']: yaml.append(f"license_link: {f['license_link']}")
    yaml += [f"base_model: {f['base']}", 'library_name: peft', 'pipeline_tag: text-generation', 'language:', '- en',
             'tags:', '- prompt-injection', '- indirect-prompt-injection', '- llm-security', '- lora', '- twin-distillation', '---']
    llama = f['tag'] == 'l_role_v13'
    if llama:
        usage = f'''```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("{f['base']}", torch_dtype="bfloat16", device_map="auto")
model = PeftModel.from_pretrained(base, "{f['repo_id']}").eval()
tok = AutoTokenizer.from_pretrained("{f['tokenizer']}")   # its chat template has the input role
SYSTEM = "{TASK_SYSTEM}"
messages = [{{"role": "system", "content": SYSTEM}},
            {{"role": "user", "content": "Summarise the key information in the document below."}},
            {{"role": "input", "content": document_text}}]   # the untrusted document goes in the input role
ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=160, do_sample=False)
print(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
```
The base model is gated: accept the Llama 3.1 licence on its Hugging Face page first. Remove Llama special tokens (`<|start_header_id|>`, `<|end_header_id|>`, `<|eot_id|>`, `<|begin_of_text|>`) from the document before passing it, as the paper's code does. The defence was trained and measured in this two-role layout; the same weights run with the document inside the user turn are a different, weaker defence (paper, Section 7.11).'''
    else:
        usage = f'''```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("{f['base']}", torch_dtype="bfloat16", device_map="auto")
model = PeftModel.from_pretrained(base, "{f['repo_id']}").eval()
tok = AutoTokenizer.from_pretrained("{f['base']}")
SYSTEM = "{TASK_SYSTEM}"
prompt = "Summarise the key information in the document below.\\n\\n" + document_text   # task, blank line, document
messages = [{{"role": "system", "content": SYSTEM}}, {{"role": "user", "content": prompt}}]
ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=160, do_sample=False)
print(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
```
The paper's models were run with the base in 4-bit quantisation and greedy decoding; bfloat16 is fine for use. The defence was trained and measured in this one-user-turn layout (paper, Section 7.11).'''
    wb = (f"- A white-box attacker with the weights (GCG) broke this adapter on {n['gcg']} of 25 documents in the paper's run (undefended base {n['gcg_base']}); the paper's Qwen adapters are broken on 0.440 and 0.760, as are the published defences on those bases. Do not rely on any adapter alone against an attacker who holds the weights."
          if llama else
          f"- A white-box attacker with the weights (GCG) still breaks this adapter on {n['gcg']} of 25 documents (undefended base {n['gcg_base']}); the published defences on the same base are broken as well. Do not rely on this adapter alone against an attacker who holds the weights.")
    built = '\n\n**Built with Llama.** Llama 3.1 is licensed under the Llama 3.1 Community License, Copyright Meta Platforms, Inc. All Rights Reserved.' if llama else ''
    hard = " (the hardened phase of this adapter used attacked wordings written by the paper's adaptive rewriter; paper, Section 5.4)" if f['tag'] == 'q7_v12' else (" (none on Llama)" if llama else '')
    lic = {'q3_v13': 'This adapter is a derivative of Qwen2.5-3B-Instruct and is released under the Qwen Research License of its base model (research use; see the licence link above).',
           'q7_v12': 'This adapter is a derivative of Qwen2.5-7B-Instruct and is released under the Apache 2.0 licence of its base model.',
           'l_role_v13': 'This adapter is a derivative of Llama-3.1-8B-Instruct and is released under the Llama 3.1 Community License of its base model.'}[f['tag']]
    body = f'''# {f['name']}

A LoRA adapter (rank 16) for **{f['base']}** trained with **twin distillation**, a fine-tuning recipe against indirect prompt injection: the model learns to judge an instruction by where it came from, not by what it says. It is the final {f['base'].split('/')[-1]} model of the paper *{PAPER}* (Md Anas Biswas, 2026; code at {REPO_URL}). Nothing else about the model changes: same base weights, same chat template, one small adapter.{built}

## What it does

- Follows the user's instructions and ignores instructions that sit inside the document it is asked to read, including instructions dressed as a fake user turn, a fake end of task, an authority claim and the other standard attack forms.
- Keeps the document's text when asked to copy, count, list, edit or translate it: an instruction-like sentence in the document is carried through like any other sentence, neither obeyed nor deleted.
- Trained for the summarisation framing (a task sentence over a document) and measured on that framing, on SecFid's counting, extraction, translation and editing tasks, and on AgentDojo tool outputs read as text. A deployment in another framing needs its own measurement.

## Prompt layout

{f['layout']}. System prompt used in training and evaluation: `{TASK_SYSTEM}`

## How to load

{usage}

## Headline numbers (paper, Tables 4 and 13; report files at commit {COMMIT_NUMBERS})

| measure | this adapter | undefended base |
|---|---|---|
| BIPIA injections followed, fixed attacker, 13 wordings (lower is better) | {n['fixed13']} | {n['fixed13_base']} |
| BIPIA injections followed, adaptive attacker, 13 attempts | {n['adaptive']} | {n['adaptive_base']} |
| BIPIA injections followed, combined | {n['combined']} | {n['combined_base']} |
| sealed S2: fresh injected instructions followed (rule-decided, 500 rows) | {n['s2']} | {n['s2_base']} |
| sealed S4: the user's own instructions followed on fresh documents (240; higher is better) | {n['s4']} | {n['s4_base']} |
| white-box GCG attack, share of 25 documents broken | {n['gcg']} | {n['gcg_base']} |

## Limits

{wb}
- The defence is tied to the prompt layout it was trained in (above). Run it in that layout.
- Residual attack success against the fixed attacker comes mostly from rewrites that replace most of the host document (paper, Section 7.1).
- One training run per model; the paper reports confidence intervals for every number above.

## Training

Four stages on one adapter: twin distillation (an on-policy distillation loss on attacked prompts, each paired with a twin in which the same instruction comes from the user), a literal phase, processing and task phases{hard}, and a forged-turn phase. Full recipe, data and settings: paper, Section 5 and Table 3; code: notebook {f['notebook']} and `src/` in the repository.

## Licence

{lic}

## Citation

Md Anas Biswas. {PAPER}. 2026. Code: {REPO_URL}.
'''
    return '\n'.join(yaml) + '\n\n' + body

CARD_DIR = Path('reports/release/cards'); CARD_DIR.mkdir(parents=True, exist_ok=True)
for f in FINALS:
    f['card'] = card(f)
    (CARD_DIR / f['name']).mkdir(exist_ok=True)
    (CARD_DIR / f['name'] / 'README.md').write_text(f['card'], encoding='utf-8')
    print(f['repo_id'], '| card', len(f['card'].split()), 'words |', f['num'])
print('Stage C done: cards written under', CARD_DIR)

In [ ]:
# === Stage D. Preview: the first card in full, and what will be uploaded ===
print(FINALS[0]['card'])
print('=' * 100)
for f in FINALS:
    print(f"{f['repo_id']}: " + ', '.join(f"{p.name} ({p.stat().st_size/1e6:.1f} MB)" for p in f['files']) + ' + README.md')
print('If the card reads right, run Stage E. Nothing has been uploaded yet.')

In [ ]:
# === Stage E. Upload each adapter with its card, then verify by downloading the config back ===
import shutil, tempfile
RELEASE = []
for f in FINALS:
    url = api.create_repo(repo_id=f['repo_id'], repo_type='model', private=False, exist_ok=True)
    with tempfile.TemporaryDirectory() as td:
        for p in f['files']: shutil.copy2(p, Path(td) / p.name)
        (Path(td) / 'README.md').write_text(f['card'], encoding='utf-8')
        info = api.upload_folder(repo_id=f['repo_id'], folder_path=td, repo_type='model',
                                 commit_message=f"Twin distillation adapter ({f['tag']}, notebook {f['notebook']}); numbers from {COMMIT_NUMBERS}")
    sha = getattr(info, 'oid', None) or str(info)
    # verify: the config that came back is byte-identical to the one on Drive
    back = hf_hub_download(repo_id=f['repo_id'], filename='adapter_config.json', force_download=True)
    assert sha256(back) == f['sha']['adapter_config.json'], f"adapter_config.json differs after upload for {f['repo_id']}"
    listed = set(api.list_repo_files(f['repo_id']))
    for p in f['files']: assert p.name in listed, f"{p.name} missing on the Hub for {f['repo_id']}"
    RELEASE.append(dict(tag=f['tag'], repo_id=f['repo_id'], url=f"https://huggingface.co/{f['repo_id']}", hub_commit=sha, base=f['base'],
                        license=f['license_name'] or f['license'], adapter_dir=f['adapter'], size_mb=round(f['size_mb'], 1),
                        sha256_weights=next(v for k, v in f['sha'].items() if k.startswith('adapter_model')), numbers_commit=COMMIT_NUMBERS,
                        released=time.strftime('%Y-%m-%d')))
    print('uploaded and verified:', RELEASE[-1]['url'])
print('Stage E done')

In [ ]:
# === Stage F. Record the release in the repository (CSV + the cards already written) ===
Path('reports/release').mkdir(exist_ok=True)
pd.DataFrame(RELEASE).to_csv(f'reports/release/adapters_{NB}.csv', index=False)
print(pd.DataFrame(RELEASE)[['tag', 'url', 'hub_commit', 'license', 'size_mb']].to_string(index=False))
print()
print('For the manuscript (Data and code availability): the trained adapters are at')
for r in RELEASE: print('  ', r['url'])
try:
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
    from reslog import log_result
    log_result(f'nb{NB}: the three final adapters released on Hugging Face', '\n'.join(f"{r['tag']} -> {r['url']} (hub commit {r['hub_commit'][:8]}, {r['license']}, {r['size_mb']} MB)" for r in RELEASE) + f"\nnumbers on the cards from commit {COMMIT_NUMBERS}")
except Exception as e:
    print('results log not updated:', e)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir('/content/drive/MyDrive/PICALIB_Research/picalib-research')
r = subprocess.run(["python", "tools/commit_cell.py", "nb83: release the three final adapters on Hugging Face (model cards under reports/release/cards, URLs and hub commits in reports/release/adapters_83.csv)"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)